In [ ]:
# 🌍 Limit Order Transformer Training (Sniper Edition)
# This model is specifically trained to predict "Dip-then-Rip" setups for accurate Limit Orders.
# It ignores "Market Buy" pumps and focuses on entries that offer a discount (Limit Fill).

# 1. Setup & Dependencies
!pip install -q pybit pandas numpy torch matplotlib scikit-learn yfinance

import os
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import yfinance as yf
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Feature Engineering (Limit Order Logic)
def prepare_features(df):
    df = df.copy()
    
    # --- 1. Trend & Momentum ---
    df['returns'] = df['close'].pct_change()
    df['log_ret'] = np.log(df['close'] / df['close'].shift(1))
    df['mom_1h'] = df['close'].pct_change(12)
    df['mom_4h'] = df['close'].pct_change(48)
    
    # --- 2. Volatility ---
    df['volatility'] = df['log_ret'].rolling(20).std()
    # Save RAW volatility for Loss Function (Before Normalization)
    df['volatility_raw'] = df['volatility'].copy()
    
    df['atr'] = (np.maximum(df['high'] - df['low'], 
                 np.maximum(abs(df['high'] - df['close'].shift(1)), 
                            abs(df['low'] - df['close'].shift(1))))).rolling(14).mean()
    df['atr_pct'] = df['atr'] / df['close']
    
    # --- 3. Oscillators ---
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))
    
    ema12 = df['close'].ewm(span=12).mean()
    ema26 = df['close'].ewm(span=26).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9).mean()
    df['macd_hist'] = macd - signal
    
    # --- 4. Relative Position ---
    sma20 = df['close'].rolling(20).mean()
    std20 = df['close'].rolling(20).std()
    df['bb_width'] = (4 * std20) / sma20
    df['bb_pos'] = (df['close'] - (sma20 - 2*std20)) / (4*std20)
    
    ema50 = df['close'].ewm(span=50).mean()
    df['dist_ema50'] = (df['close'] / ema50) - 1
    
    # --- 5. Volume ---
    vol_ma20 = df['vol'].rolling(20).mean()
    df['vol_ratio'] = df['vol'] / (vol_ma20 + 1e-8)
    
    # --- 6. LIMIT ORDER TARGETS (The Secret Sauce) ---
    # We want to find setups where price DIPS to fill a limit, then RIPS to profit.
    horizon = 12 # 1 Hour window
    
    # Look ahead for Highs and Lows
    indexer = pd.api.indexers.FixedForwardWindowIndexer(window_size=horizon)
    df['future_low'] = df['low'].rolling(window=indexer).min()
    df['future_high'] = df['high'].rolling(window=indexer).max()
    
    # Settings for "Perfect Trade"
    LIMIT_OFFSET = 0.002 # We place limit 0.2% BELOW close
    TAKE_PROFIT = 0.008  # We want 0.8% profit from close (approx 1.0% from entry)
    STOP_LOSS = 0.005    # We risk 0.5% from close
    
    # Buy Signal Logic:
    # 1. Price MUST dip to (Close - 0.2%) to fill our order.
    # 2. Price MUST rise to (Close + 0.8%) to hit TP.
    # 3. Price MUST NOT drop below (Close - 0.5%) at any point (Stop Loss).
    buy_signal = (
        (df['future_low'] <= df['close'] * (1 - LIMIT_OFFSET)) & 
        (df['future_high'] >= df['close'] * (1 + TAKE_PROFIT)) &
        (df['future_low'] > df['close'] * (1 - STOP_LOSS))
    )
    
    # Sell Signal Logic:
    # 1. Price MUST spike to (Close + 0.2%) to fill our order.
    # 2. Price MUST drop to (Close - 0.8%) to hit TP.
    # 3. Price MUST NOT rise above (Close + 0.5%) at any point.
    sell_signal = (
        (df['future_high'] >= df['close'] * (1 + LIMIT_OFFSET)) &
        (df['future_low'] <= df['close'] * (1 - TAKE_PROFIT)) &
        (df['future_high'] < df['close'] * (1 + STOP_LOSS))
    )
    
    df['target'] = 1 # Neutral
    df.loc[buy_signal, 'target'] = 2 # Buy
    df.loc[sell_signal, 'target'] = 0 # Sell
    
    # Cleanup
    df = df.dropna()
    
    # Feature Selection
    features = ['log_ret', 'volatility', 'rsi', 'macd_hist', 'bb_width', 'bb_pos', 
                'atr_pct', 'vol_ratio', 'dist_ema50', 'mom_1h', 'mom_4h', 'returns']
                
    # Normalize
    for col in features:
        df[col] = (df[col] - df[col].mean()) / df[col].std()
        
    return df, features

# 3. Transformer Model (Standard)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class LimitOrderTransformer(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=4, num_layers=3, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=d_model*4, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.direction_head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 3)
        )
        
    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)
        return self.direction_head(x)

# 4. Robust Loss (FIXED: Strictly Positive)
class RobustLoss(nn.Module):
    def __init__(self, volatility_penalty=1.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(reduction='none')
        self.volatility_penalty = volatility_penalty
        
    def forward(self, pred, target, volatility):
        loss = self.ce(pred, target)
        
        # Normalize volatility to 0-1 range using Sigmoid to guarantee positivity
        # We want higher volatility to have higher weight
        # volatility input is raw std dev (e.g., 0.005, 0.02)
        # We scale it up so sigmoid has a nice curve
        vol_scaled = volatility * 100 
        vol_weight = torch.sigmoid(vol_scaled) 
        
        # Weight = 1 + (0 to 1 * penalty)
        # This ensures weight is always >= 1.0
        weight = 1.0 + (vol_weight * self.volatility_penalty)
        
        weighted_loss = loss * weight
        return weighted_loss.mean()

# 5. Data Fetching (Yahoo Finance - More Reliable)
def get_top_liquid_symbols(n=100):
    # Hardcoded top list to avoid API dependency
    return [
        "BTC-USD", "ETH-USD", "SOL-USD", "XRP-USD", "ADA-USD", "DOGE-USD", "AVAX-USD", "DOT-USD", "TRX-USD", "LINK-USD",
        "MATIC-USD", "SHIB-USD", "LTC-USD", "BCH-USD", "ATOM-USD", "UNI-USD", "XLM-USD", "ETC-USD", "FIL-USD", "HBAR-USD",
        "APT-USD", "NEAR-USD", "VET-USD", "QNT-USD", "AAVE-USD", "ALGO-USD", "GRT-USD", "FTM-USD", "SAND-USD", "EOS-USD",
        "MANA-USD", "THETA-USD", "EGLD-USD", "XTZ-USD", "AXS-USD", "FLOW-USD", "CHZ-USD", "RUNE-USD", "ZEC-USD", "KAVA-USD"
    ]

def fetch_data(symbol):
    try:
        # Fetch 59 days of 5m data (60d sometimes hits limits)
        # auto_adjust=True simplifies columns (Close vs Adj Close)
        df = yf.download(symbol, period="59d", interval="5m", progress=False, auto_adjust=True)
        
        if len(df) < 100: 
            # print(f"   ⚠️ {symbol}: Too few rows ({len(df)})")
            return None
        
        # Handle MultiIndex columns (common in new yfinance)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
            
        # Standardize columns
        df = df.reset_index()
        df.columns = [str(c).lower() for c in df.columns] # Ensure string for .lower()
        
        # Rename common variations
        rename_map = {
            'datetime': 'ts', 'date': 'ts', 
            'volume': 'vol', 'adj close': 'close'
        }
        df = df.rename(columns=rename_map)
        
        # Check required columns
        required = ['open', 'high', 'low', 'close', 'vol']
        if not all(col in df.columns for col in required):
            # print(f"   ⚠️ {symbol}: Missing columns. Found: {df.columns.tolist()}")
            return None
            
        # Ensure numeric
        for c in required:
            df[c] = pd.to_numeric(df[c], errors='coerce')
            
        df = df.dropna()
        return df
    except Exception as e:
        print(f"   ❌ {symbol} Error: {e}")
        return None

print("⏳ Fetching Data...")
symbols = get_top_liquid_symbols(40) # Start with 40 for speed
all_data = []

with ThreadPoolExecutor(max_workers=10) as executor:
    future_to_sym = {executor.submit(fetch_data, sym): sym for sym in symbols}
    for future in as_completed(future_to_sym):
        sym = future_to_sym[future]
        df = future.result()
        if df is not None and len(df) > 1000:
            try:
                processed_df, _ = prepare_features(df)
                # Balance classes per symbol
                buys = processed_df[processed_df['target'] == 2]
                sells = processed_df[processed_df['target'] == 0]
                neutrals = processed_df[processed_df['target'] == 1]
                
                # Undersample neutrals
                n_samples = min(len(buys), len(sells)) * 2
                if n_samples > 0:
                    neutrals = neutrals.sample(n=min(len(neutrals), n_samples))
                    balanced = pd.concat([buys, sells, neutrals])
                    all_data.append(balanced)
                    print(f"   ✅ {sym}: {len(balanced)} samples")
            except Exception as e:
                print(f"   ⚠️ {sym} Processing Error: {e}")
                pass

if not all_data:
    print("❌ CRITICAL: No training data collected!")
    raise ValueError("No training data collected.")

full_df = pd.concat(all_data)
print(f"🔥 Total Training Samples: {len(full_df)}")

# 6. Dataset
class CryptoDataset(Dataset):
    def __init__(self, df, seq_len=72):
        self.features = df[['log_ret', 'volatility', 'rsi', 'macd_hist', 'bb_width', 'bb_pos', 
                            'atr_pct', 'vol_ratio', 'dist_ema50', 'mom_1h', 'mom_4h', 'returns']].values
        self.targets = df['target'].values
        # Use RAW volatility for loss weighting
        self.volatility = df['volatility_raw'].values
        self.seq_len = seq_len
        
    def __len__(self):
        return len(self.features) - self.seq_len
        
    def __getitem__(self, idx):
        x = torch.FloatTensor(self.features[idx:idx+self.seq_len])
        y = torch.LongTensor([self.targets[idx+self.seq_len-1]])[0]
        vol = torch.FloatTensor([self.volatility[idx+self.seq_len-1]])[0]
        return x, y, vol

# Split
train_size = int(len(full_df) * 0.8)
train_df = full_df.iloc[:train_size]
val_df = full_df.iloc[train_size:]

train_dataset = CryptoDataset(train_df)
val_dataset = CryptoDataset(val_df)

# Sampler
class_counts = np.bincount(train_df['target'])
class_weights = 1. / class_counts
sample_weights = class_weights[train_df['target'].values[72:]]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=64)

# 7. Training
model = LimitOrderTransformer(input_dim=12).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-3) # Lower LR
criterion = RobustLoss(volatility_penalty=2.0)

epochs = 30
best_val_loss = float('inf')

print("🚀 Starting Limit Order Model Training...")
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for x, y, vol in train_loader:
        x, y, vol = x.to(device), y.to(device), vol.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y, vol)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y, vol in val_loader:
            x, y, vol = x.to(device), y.to(device), vol.to(device)
            pred = model(x)
            loss = criterion(pred, y, vol)
            val_loss += loss.item()
            _, predicted = torch.max(pred.data, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()
            
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    acc = 100 * correct / total
    
    print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}, Val Acc={acc:.2f}%")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "limit_transformer_best.pt")
        print("   ✅ Saved Best Limit Model")

# 8. Download
from google.colab import files
files.download('limit_transformer_best.pt')